# Конспект: линейные модели, деревья и лес

**Олимпиадный маршрут:** выбрать цель и метрику → убрать утечки → разделить данные → обучить подготовку и модели только на train → сравнить на validation → предсказать test.

| Ответ | Быстрая модель | Если нужна нелинейность |
|---|---|---|
| Число | `LinearRegression`, `Ridge`, `Lasso` | `DecisionTreeRegressor`, `RandomForestRegressor` |
| Класс | `LogisticRegression` | `DecisionTreeClassifier`, `RandomForestClassifier` |

Линейная регрессия выдаёт число; логистическая — вероятность класса. Дерево задаёт последовательность условий `признак ≤ порог`; лес усредняет много деревьев. Лес не гарантирует победу: выбирайте по валидации.

## 1. Разбиение и метрики

| Задача | Смотреть на validation | Замечание |
|---|---|---|
| Регрессия | `MAE` (меньше лучше), `RMSE`, `R²` | MAE — средняя ошибка в единицах ответа; R² может быть отрицательным |
| Классификация | `F1`, `precision`, `recall`, `accuracy` | При дисбалансе accuracy может обмануть |
| Вероятности | `log_loss` | Оценивает качество вероятностей, а не только классов |

`train_test_split(..., stratify=y)` сохраняет доли классов; для регрессии `stratify` обычно не нужен. Test без ответов не участвует в выборе модели. ID, целевой столбец и признаки из будущего удаляйте из `X`.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.metrics import f1_score, mean_absolute_error, accuracy_score, log_loss

## 2. Классификация: полный короткий шаблон

Пример из Titanic. Выбираем только признаки, известные **до** исхода: класс, возраст, цену, пол. `Pipeline` учит медиану, кодирование и масштаб только на `X_train`; неизвестные категории при предсказании не ломают модель. Деревьям масштабирование не требуется.

In [ ]:
df = pd.read_csv('../class08: data visualization/titanic.csv')
X = df[['pclass', 'age', 'fare', 'sex']].copy()
y = df['survived']
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42)

def prep(scale=False):
    num = ['pclass', 'age', 'fare']
    cat = ['sex']
    num_steps = [('missing', SimpleImputer(strategy='median'))]
    if scale:
        num_steps.append(('scale', StandardScaler()))
    return ColumnTransformer([
        ('num', Pipeline(num_steps), num),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat),
    ])

classifiers = {
    'baseline': Pipeline([('prep', prep()), ('model', DummyClassifier(strategy='most_frequent'))]),
    'logistic': Pipeline([('prep', prep(scale=True)), ('model', LogisticRegression(max_iter=2000))]),
    'tree': Pipeline([('prep', prep()), ('model', DecisionTreeClassifier(max_depth=4, min_samples_leaf=5, random_state=42))]),
    'forest': Pipeline([('prep', prep()), ('model', RandomForestClassifier(n_estimators=150, min_samples_leaf=3, random_state=42))]),
}
for name, model in classifiers.items():
    model.fit(X_train, y_train)
    print(name, 'F1 =', round(f1_score(y_valid, model.predict(X_valid)), 3))

`predict` выдаёт классы, `predict_proba(... )[:, 1]` — вероятность **класса 1**. Если стоимость ошибок различается, подберите порог на validation и проверьте метрику; не выбирайте его по test.

In [ ]:
model = classifiers['logistic']
prob = model.predict_proba(X_valid)[:, 1]
pred = (prob >= 0.5).astype(int)
print('accuracy:', round(accuracy_score(y_valid, pred), 3))
print('F1:', round(f1_score(y_valid, pred), 3))
print('log loss:', round(log_loss(y_valid, prob), 3))

## 3. Регрессия: те же шаги

На числовых признаках подготовка короче. Сравнивайте модели на **одном** разбиении; среднее значение цели — полезный базовый уровень.

In [ ]:
from sklearn.datasets import load_diabetes

Xr, yr = load_diabetes(return_X_y=True, as_frame=True)
Xr_train, Xr_valid, yr_train, yr_valid = train_test_split(
    Xr, yr, test_size=0.25, random_state=42)
regressors = {
    'baseline': DummyRegressor(strategy='mean'),
    'ridge': Pipeline([('scale', StandardScaler()), ('model', Ridge(alpha=1.0))]),
    'tree': DecisionTreeRegressor(max_depth=4, min_samples_leaf=10, random_state=42),
    'forest': RandomForestRegressor(n_estimators=150, min_samples_leaf=4, random_state=42),
}
for name, model in regressors.items():
    model.fit(Xr_train, yr_train)
    print(name, 'MAE =', round(mean_absolute_error(yr_valid, model.predict(Xr_valid)), 2))

## 4. Параметры, которые реально менять

| Модель | Параметр | Эффект |
|---|---|---|
| `Ridge`, `Lasso` | `alpha` ↑ | сильнее штраф за большие веса; Lasso может занулить часть весов |
| `LogisticRegression` | `C` ↓ | сильнее регуляризация (направление **обратное** `alpha`) |
| Дерево/лес | `max_depth` ↓, `min_samples_leaf` ↑ | проще модель, меньше риск запомнить шум |
| Лес | `n_estimators` ↑ | обычно стабильнее, но медленнее |
| Лес | `max_features` | сколько признаков проверять в каждом узле |
| Все случайные модели | `random_state=42` | повторяемость результата |

**Масштаб:** Ridge, Lasso и LogisticRegression обычно требуют `StandardScaler`; дереву и лесу он не нужен. Для категорий в sklearn используйте `OneHotEncoder(handle_unknown='ignore')`, а не случайные числа вместо названий.

## 5. Быстрая диагностика

```python
print(export_text(classifiers['tree'].named_steps['model'],
                  feature_names=classifiers['tree'].named_steps['prep'].get_feature_names_out().tolist()))
```

- Train сильно лучше validation → модель запомнила шум; ограничьте дерево/лес или усилите регуляризацию.
- Все модели дают невероятное качество → проверьте ID, дубликаты и признаки из будущего.
- `fit` падает на `NaN`/строках → заполните пропуски, закодируйте категории внутри `Pipeline`.
- `predict_proba` содержит два столбца → проверьте `model.classes_` (у Pipeline: `model.named_steps['model'].classes_`).
- Подобрать силу штрафа: `Ridge(alpha=0.1)` / `Lasso(alpha=0.1)` для числа; `LogisticRegression(C=0.1, max_iter=2000)` для класса. Сравнивайте несколько значений на validation или кросс-валидации.

### Шаблон файла для сдачи

```python
best = classifiers['forest']       # модель и параметры уже выбраны на validation
best.fit(X, y)                     # теперь все размеченные строки
test = pd.read_csv('test.csv')
ids = test['id'].copy()            # имя ID замените на своё
proba = best.predict_proba(test[X.columns])[:, 1]
submission = pd.DataFrame({'id': ids, 'prediction': proba})
assert len(submission) == len(test) and submission.notna().all().all()
submission.to_csv('submission.csv', index=False)
```

Колонки и тип ответа (`proba` или класс) должны точно соответствовать условию задачи.

- Перед сдачей: после выбора параметров обучите выбранный Pipeline на **всех размеченных train**; для test сохраните исходный порядок и ID, файл пишите с `index=False`.